# IPF V3 — QC bridge state exporter

Run in QuantConnect Research. It prints one `QC_BRIDGE_BUNDLE,...` line containing the frozen rolling state through QC's latest available session. Do not edit the signal logic.

In [ ]:
from AlgorithmImports import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json

qb = QuantBook()
qb.set_time_zone(TimeZones.NEW_YORK)

qqq = qb.add_equity("QQQ", Resolution.MINUTE).symbol

end = datetime.utcnow()
start = end - timedelta(days=650)

h = qb.history(qqq, start, end, Resolution.MINUTE)

def normalize_history(hist):
    if hist is None or len(hist) == 0:
        return pd.DataFrame(columns=["time", "close"])
    df = hist.copy().reset_index()
    df.columns = [str(c).lower() for c in df.columns]
    if "time" not in df.columns:
        candidates = [c for c in df.columns if "time" in c or "date" in c]
        if not candidates:
            raise RuntimeError(f"Could not identify time column: {list(df.columns)}")
        df = df.rename(columns={candidates[0]: "time"})
    if "close" not in df.columns:
        raise RuntimeError(f"Could not identify close column: {list(df.columns)}")
    df["time"] = pd.to_datetime(df["time"])
    try:
        if df["time"].dt.tz is not None:
            df["time"] = df["time"].dt.tz_convert("America/New_York").dt.tz_localize(None)
    except Exception:
        pass
    return df[["time", "close"]].sort_values("time").reset_index(drop=True)

q = normalize_history(h)
if q.empty:
    raise RuntimeError("QC_BRIDGE_ERROR,no_history")

q["date"] = q["time"].dt.normalize()

q1545 = (
    q[(q["time"].dt.hour == 15) & (q["time"].dt.minute == 45)]
    [["date", "close"]]
    .drop_duplicates("date", keep="last")
    .rename(columns={"close": "px1545"})
)
q1600 = (
    q[(q["time"].dt.hour == 16) & (q["time"].dt.minute == 0)]
    [["date", "close"]]
    .drop_duplicates("date", keep="last")
    .rename(columns={"close": "close1600"})
)

daily = q1545.merge(q1600, on="date", how="inner").sort_values("date").reset_index(drop=True)
daily["prev_close"] = daily["close1600"].shift(1)
daily["signal_return"] = daily["px1545"] / daily["prev_close"] - 1.0
daily = daily.dropna(subset=["signal_return"]).reset_index(drop=True)

if len(daily) < 252:
    raise RuntimeError(f"QC_BRIDGE_ERROR,insufficient_sessions={len(daily)}")

asof = daily.iloc[-1]
prior = daily["signal_return"].iloc[-252:].astype(float).tolist()

overlap = []
for _, r in daily.tail(7).iterrows():
    overlap.append({
        "date": r["date"].date().isoformat(),
        "px1545": float(r["px1545"]),
        "close1600": float(r["close1600"]),
        "signal_return": float(r["signal_return"]),
    })

bundle = {
    "asof_date": asof["date"].date().isoformat(),
    "prev_close": float(asof["close1600"]),
    "prior_returns": prior,
    "prior_mean": float(np.mean(prior)),
    "prior_sd_sample": float(np.std(prior, ddof=1)),
    "overlap": overlap,
}

print(
    f"QC_BRIDGE_ACCESS,rows={len(q)},max_time={q['time'].max()},"
    f"asof={bundle['asof_date']},prior_n={len(prior)}"
)
print("QC_BRIDGE_BUNDLE," + json.dumps(bundle, separators=(",", ":")))
